In [0]:
%sql
select * from read_files("/Volumes/dev/sunil/sunil_volume/ACUnsubs_Backfill_20260414_113509.csv")

In [0]:
%sql
create table dev.sunil.day2table_sql
TBLPROPERTIES ('delta.columnMapping.mode' = 'name')
as 
select * from read_files("/Volumes/dev/sunil/sunil_volume/ACUnsubs_Backfill_20260414_113509.csv")

In [0]:
%sql
select * from dev.sunil.day2table_sql

In [0]:
%sql
desc extended dev.sunil.day2table_sql

In [0]:
%sql
desc detail dev.sunil.day2table_sql

In [0]:
%sql
select * from read_files("/Volumes/dev/naval/raw/drivers.json")

In [0]:
%sql
create table dev.sunil.jsonfile as 
select * from read_files("/Volumes/dev/naval/raw/drivers.json")

In [0]:
%sql
create or replace table dev.sunil.jsonfile as 
select *,current_timestamp()  as ingestion_date, _metadata.file_name as path from read_files("/Volumes/dev/naval/raw/drivers.json")

In [0]:
%sql
select * from dev.sunil.jsonfile

In [0]:
from pyspark.sql.functions import *

In [0]:
df=spark.read.csv("/Volumes/dev/naval/raw/financial_dataset.csv",header=True,inferSchema=True)

In [0]:
df1=df.withColumn("ingestion_date",current_timestamp()).withColumn("path",col("_metadata.file_name"))

In [0]:
df1.write.mode("overwrite").saveAsTable("dev.sunil.financial_spark")

In [0]:
df=spark.read.csv("/Volumes/dev/naval/raw/financial_dataset.csv",header=True,inferSchema=True)
df1=df.withColumn("ingestion_date",current_timestamp()).withColumn("path",col("_metadata.file_name"))
df1.write.mode("overwrite").saveAsTable("dev.sunil.financial_spark")

In [0]:
(# reading
spark
 .read
 .csv("/Volumes/dev/naval/raw/financial_dataset.csv",header=True,inferSchema=True)

 # tranformation
 .withColumn("ingestion_date",current_timestamp())
 .withColumn("path",col("_metadata.file_name"))

 # writing / Loading
 .write
 .mode("overwrite")
 .saveAsTable("dev.sunil.financial_spark")
 )

In [0]:
input_path="/Volumes/dev/naval/raw/financial_dataset.csv"
catalog="dev"
schema="sunil"
table_name="financial_spark"

In [0]:
def add_columns(df):
    return df.withColumn("ingestion_date",current_timestamp()).withColumn("path",col("_metadata.file_name"))

In [0]:
df=(spark.read.csv(input_path,header=True,inferSchema=True))
df1=add_columns(df)
df1.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.{table_name}")

In [0]:
%sql
select * from dev.sunil.financial_spark

In [0]:
df=spark.table("dev.sunil.financial_spark")

In [0]:
df.display()

In [0]:
df=(spark.read.csv(input_path,header=True,inferSchema=True))

In [0]:
#df.select("*")
df.select("TransactionID","AmountUSD").display()

In [0]:
df.filter("AmountUSD>1000").display()

In [0]:
df.orderBy(col("TransactionType").desc()).display()

In [0]:
df.withColumnRenamed("TransactionID","transaction_id").withColumnRenamed("CustomerID","customer_id")

In [0]:
df.withColumnsRenamed({"TransactionID":"transaction_id","CustomerID":"customer_id"})

In [0]:
df.printSchema()

In [0]:
df_lower = df.toDF(*[col_name.lower() for col_name in df.columns])
df_lower.display()

In [0]:
df_lower.groupBy("branch").count().display()

In [0]:
df_lower.withColumn("env",lit("dev")).display()

In [0]:
df_lower.groupBy("branch").agg(count("*"), 
                                       sum("amountusd").alias("sum"),
                                       min("amountusd").alias("min")).display()